# Tokenizer tutorial: HF byte-level BPE

This project trains with **`HFByteBPETokenizer`** only (`hf_bpe_byte`), implemented in `src/nano_llm/tokenizer.py` using Hugging Face [`tokenizers`](https://github.com/huggingface/tokenizers).

You will:
- train a tokenizer on text with `from_text` / `build_tokenizer_from_text`
- encode and decode, including Unicode
- save and restore with `to_state` / `from_state` / `tokenizer_from_state` (checkpoint compatibility)
- optionally compare **word-boundary-aware** vs default pre-tokenization

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")
sys.path.insert(0, str(repo_root / "src"))

from nano_llm.tokenizer import HFByteBPETokenizer, build_tokenizer_from_text, tokenizer_from_state

## 1) Train on text and roundtrip

`vocab_size` caps the learned merge table. Training uses the byte-level pre-tokenizer so emoji and non-Latin scripts stay well-defined.

In [ ]:
corpus = "Hello world.\nCafé 你好 😀\n" * 50
tok = HFByteBPETokenizer.from_text(corpus, vocab_size=256, word_boundary_aware=False)
text = "Hello Café 你好"
ids = tok.encode(text)
decoded = tok.decode(ids)
print("vocab_size:", tok.vocab_size)
print("ids length:", len(ids))
print("roundtrip ok:", decoded == text)
print("state type:", tok.to_state()["type"])

## 2) Checkpoint-style state

Checkpoints store `tokenizer_state` with `type: hf_bpe_byte` and `tokenizer_json`. Restoring must use the same JSON so ids align with training.

In [ ]:
state = tok.to_state()
tok2 = tokenizer_from_state(state)
assert tok2.encode(text) == ids
print("restored ok")

## 3) `build_tokenizer_from_text`

Same as `from_text`, with explicit kwargs used by `nano_llm.train` when building from corpus strings.

In [ ]:
tok3 = build_tokenizer_from_text(corpus, bpe_vocab_size=256, bpe_word_boundary_aware=False)
print("pad_id:", tok3.pad_id)

## 4) Word-boundary-aware BPE (optional)

With `word_boundary_aware=True`, the byte pre-tokenizer uses a mode that discourages merges across whitespace boundaries (often more interpretable subwords on small corpora). Try both on your data and compare validation loss.

In [ ]:
tw = HFByteBPETokenizer.from_text(corpus, vocab_size=256, word_boundary_aware=True)
tn = HFByteBPETokenizer.from_text(corpus, vocab_size=256, word_boundary_aware=False)
s = "to be or not to be"
print("wb-aware token count:", len(tw.encode(s)))
print("default token count :", len(tn.encode(s)))

## 5) Training CLI

The trainer fixes `tokenizer_type` to `hf_bpe_byte` and fits the tokenizer on train+val text unless you resume from a checkpoint that already contains `tokenizer_state`.

```bash
python scripts/train.py --tokenizer-type hf_bpe_byte --bpe-vocab-size 256
```

See also: `docs/imdb_sentiment_tutorial.ipynb` for end-to-end IMDB formatting.